### Análisis y Procesamiento de Señales
# Trabajo Práctico N° 3 - Desparramo Espectral, Teorema de Parseval y Zero Padding
### Pedro Joaquín Cannavo

## 1. Introducción

En este trabajo práctico se analiza el fenómeno de **desparramo espectral** (*spectral leakage*) que se produce al calcular la Transformada Discreta de Fourier (DFT/FFT) en señales de tiempo discreto.

Para estudiar este efecto, se generaron tres señales senoidales de potencia unitaria con frecuencias muy próximas entre sí, agregando una pequeña desintonía fraccionaria respecto a la resolución espectral de la FFT. Se compararon sus espectros de densidad de potencia, se comprobó la conservación de la potencia mediante el **Teorema de Parseval** y se aplicó la técnica de **zero padding**, observando cómo el agregado de muestras nulas permite interpolar el espectro y recuperar visualmente la amplitud real de la señal.

## 2. Marco Teórico y Parámetros de Diseño

### 2.1 Muestreo y Resolución Espectral
Se trabajó con un registro de $N = 1000$ muestras a una frecuencia de muestreo de $f_s = 1000\text{ Hz}$, lo que define una duración temporal de $1$ segundo. La resolución espectral básica de la FFT está dada por:

$$\Delta f = \frac{f_s}{N} = \frac{1000\text{ Hz}}{1000} = 1\text{ Hz}$$

Cuando la frecuencia de una senoidal coincide exactamente con un múltiplo entero de $\Delta f$ ($f_0 = k_0 \cdot \Delta f$ con $k_0$ entero), dentro de la ventana de 1000 muestras entra un número exacto de períodos completos. En este caso toda la energía queda contenida en ese único bin de la FFT y no hay desparramo.

En cambio, si la frecuencia no cae de forma exacta sobre un bin ($k_0$ no entero), la señal no completa un número entero de ciclos en el registro. Al calcular la FFT, la energía no puede ubicarse en un solo punto y termina desparramándose hacia los bines adyacentes.

### 2.2 Potencia Unitaria
Para lograr una potencia unitaria de $P = 1\text{ W}$ sobre una resistencia de referencia de $R = 1\,\Omega$, la tensión eficaz de la senoidal debe ser de $1\text{ V}$. De esta forma, la amplitud máxima necesaria es:

$$V_{max} = V_{rms} \sqrt{2} = \sqrt{2}\text{ V} \approx 1.414\text{ V}$$

### 2.3 Teorema de Parseval
El teorema de Parseval establece la conservación de la potencia entre el tiempo y la frecuencia. La potencia calculada a partir de las muestras temporales es:

$$P_{\text{tiempo}} = \frac{1}{N} \sum_{n=0}^{N-1} |x[n]|^2$$

y en el dominio de la frecuencia mediante la DFT de $N$ puntos resulta:

$$P_{\text{freq}} = \frac{1}{N^2} \sum_{k=0}^{N-1} |X[k]|^2$$

Esta relación asegura que, aunque una señal sufra desparramo y el pico en el gráfico parezca descender, la sumatoria total de potencia sobre todos los bines del espectro debe seguir sumando exactamente $1\text{ W}$.

### 2.4 Concepto de Zero Padding
La técnica de zero padding consiste en agregar una cantidad determinada de ceros al final del vector temporal original. Al sumar $9000$ ceros a las $1000$ muestras iniciales, el registro total pasa a tener $10000$ muestras.

Como los ceros agregados no aportan energía ni modifican la señal original, la técnica no mejora la resolución física para separar dos tonos muy próximos. Lo que hace es evaluar la transformada continua en una grilla de frecuencias diez veces más densa:

$$\Delta f_{zp} = \frac{f_s}{N_{zp}} = \frac{1000\text{ Hz}}{10000} = 0.1\text{ Hz}$$

Esto funciona como una interpolación en frecuencia, permitiendo observar la forma continua de los lóbulos y recuperar el valor real del pico que antes caía entre dos bines originales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parametros generales
N = 1000                  # Cantidad de muestras originales
fs = 1000.0               # Frecuencia de muestreo [Hz]
df = fs / N               # Resolucion espectral original: 1 Hz

R = 1.0                   # Resistencia normalizada [ohm]
vmax = np.sqrt(2)         # Amplitud pico para lograr potencia unitaria (1 W)
dc = 0.0
ph = 0.0

# Eje temporal original (0 a 1 segundo)
tt = np.arange(N) / fs

def mi_funcion_sen(vmax=np.sqrt(2), dc=0, ff=1, ph=0, nn=N, fs=fs):
    t = np.arange(nn) / fs
    x = vmax * np.sin(2 * np.pi * ff * t + ph) + dc
    return t, x

## 3. Desarrollo y Experimentación

### 3.1 Inciso a: Densidad Espectral de Potencia con Distintas Desintonías

Se generan tres senoidales alrededor de $250\text{ Hz}$ variando el factor $k_0$:
1. $k_0 = N/4 = 250$ ($f_1 = 250.0\text{ Hz}$, frecuencia sintonizada de forma exacta en un bin).
2. $k_0 = N/4 + 0.25$ ($f_2 = 250.25\text{ Hz}$, desintonía intermedia de un cuarto de bin).
3. $k_0 = N/4 + 0.5$ ($f_3 = 250.5\text{ Hz}$, desintonía máxima de medio bin).

In [ ]:
# Frecuencias solicitadas
f1 = (N / 4) * df          # 250.0 Hz (k0 = 250 - Coherente)
f2 = (N / 4 + 0.25) * df   # 250.25 Hz (k0 = 250.25 - Desintonia leve)
f3 = (N / 4 + 0.5) * df    # 250.5 Hz (k0 = 250.5 - Desintonia maxima)

tt, x1 = mi_funcion_sen(vmax=vmax, dc=dc, ff=f1, ph=ph, nn=N, fs=fs)
tt, x2 = mi_funcion_sen(vmax=vmax, dc=dc, ff=f2, ph=ph, nn=N, fs=fs)
tt, x3 = mi_funcion_sen(vmax=vmax, dc=dc, ff=f3, ph=ph, nn=N, fs=fs)

# Eje de frecuencias original (0 a fs/2 = 500 Hz)
ff_pos = np.fft.rfftfreq(N, 1/fs)

# FFT unilateral normalizada
X1 = np.fft.rfft(x1) / N
X2 = np.fft.rfft(x2) / N
X3 = np.fft.rfft(x3) / N

# Densidad Espectral de Potencia en dB (0 dB = 1 W)
X1_dB = 10 * np.log10(2 * np.abs(X1)**2 + 1e-15)
X2_dB = 10 * np.log10(2 * np.abs(X2)**2 + 1e-15)
X3_dB = 10 * np.log10(2 * np.abs(X3)**2 + 1e-15)

# Grafico 1: Inciso a - PSD Original
plt.figure(1, figsize=(11, 5))
plt.plot(ff_pos, X1_dB, 'o:', label=f'k = N/4 ({f1:.2f} Hz) - Coherente', color='#FF007F', lw=1)
plt.plot(ff_pos, X2_dB, 'o:', label=f'k = N/4 + 0.25 ({f2:.2f} Hz)', color='#00A8FF', lw=1)
plt.plot(ff_pos, X3_dB, 'o:', label=f'k = N/4 + 0.5 ({f3:.2f} Hz) - Max. desintonia', color='#00FF80', lw=1)

plt.title('Inciso a: Densidad Espectral de Potencia (PSD) - Desparramo Espectral')
plt.xlabel('Frecuencia [Hz]')
plt.ylabel('Densidad de Potencia [dB]')
plt.xlim(230, 270)       # Zoom en la zona de desparramo
plt.ylim(-60, 5)
plt.grid(True, alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Análisis de la Figura 1:**

En el gráfico se puede apreciar claramente el impacto de la desintonía en la FFT. Para el primer caso ($250.0\text{ Hz}$), como la frecuencia cae exactamente sobre un bin entero, toda la potencia queda concentrada en esa sola muestra alcanzando los $0\text{ dB}$, mientras que en todas las demás frecuencias el valor es prácticamente nulo.

En cambio, a medida que agregamos desintonía ($250.25\text{ Hz}$ y $250.5\text{ Hz}$), el pico medido desciende de forma notable (llegando a casi $-3.9\text{ dB}$ para el caso de medio bin) y comienzan a aparecer componentes en los bines adyacentes formando lóbulos laterales. Esto muestra cómo una diferencia de apenas fracciones de Hertz hace que el espectro pase de verse como un tono puro ideal a verse como una distribución ensanchada con pérdida aparente de amplitud en el pico principal.

### 3.2 Inciso b: Comprobación de Potencia Mediante la Identidad de Parseval

Calculamos la potencia media tanto en el tiempo como en la frecuencia a través de la sumatoria de Parseval para verificar la conservación de la energía.

In [ ]:
# 1. Potencia en el tiempo (promedio de las muestras al cuadrado)
P1_tiempo = np.mean(x1**2)
P2_tiempo = np.mean(x2**2)
P3_tiempo = np.mean(x3**2)

# 2. Potencia en frecuencia usando Parseval: sum(|X|^2) / N^2
P1_freq = np.sum(np.abs(np.fft.fft(x1))**2) / (N**2)
P2_freq = np.sum(np.abs(np.fft.fft(x2))**2) / (N**2)
P3_freq = np.sum(np.abs(np.fft.fft(x3))**2) / (N**2)

print("="*60)
print("INCISO B: COMPROBACION DEL TEOREMA DE PARSEVAL")
print("="*60)
print("Potencia calculada en el tiempo:")
print(f"  Senoidal 1 (250.00 Hz): {P1_tiempo:.6f} W")
print(f"  Senoidal 2 (250.25 Hz): {P2_tiempo:.6f} W")
print(f"  Senoidal 3 (250.50 Hz): {P3_tiempo:.6f} W")

print("\nPotencia calculada en frecuencia (Parseval):")
print(f"  Senoidal 1 (250.00 Hz): {P1_freq:.6f} W")
print(f"  Senoidal 2 (250.25 Hz): {P2_freq:.6f} W")
print(f"  Senoidal 3 (250.50 Hz): {P3_freq:.6f} W")
print("------------------------------------------------------------")
print("Conclusion: En los tres casos la potencia se conserva en 1 W.")
print("La caida del pico se debe a que la energia se desparramo a los bines vecinos.")
print("="*60)

**Análisis de los Resultados de Parseval:**

Los resultados numéricos demuestran de manera contundente la validez del teorema de Parseval. A pesar de que en el gráfico del inciso anterior el pico de $250.5\text{ Hz}$ parecía haber perdido potencia respecto al de $250.0\text{ Hz}$, al sumar el aporte de todos los bines del espectro la potencia total resulta ser exactamente $1.000000\text{ W}$ en todos los casos.

Esto confirma que la energía no desaparece en el proceso de discretización frecuencial: la aparente pérdida de nivel se debe únicamente a que la energía que antes estaba concentrada en un único bin ahora se encuentra desparramada entre varias frecuencias adyacentes.

### 3.3 Inciso c: Repetición del Experimento Mediante Zero Padding

Se repite el análisis agregando $9 \times N = 9000$ ceros al final de cada señal (obteniendo un total de $10000$ muestras). Con esta nueva longitud, la separación entre puntos en frecuencia pasa a ser de $\Delta f_{zp} = 0.1\text{ Hz}$.

In [ ]:
# Agregamos 9*N ceros al final de cada senal (largo total = 10*N = 10000 muestras)
cant_ceros = 9 * N
N_zp = N + cant_ceros     # N_zp = 10000
df_zp = fs / N_zp         # Nueva grilla de frecuencias: df_zp = 0.1 Hz

# Padding con numpy
x1_zp = np.pad(x1, (0, cant_ceros), mode='constant')
x2_zp = np.pad(x2, (0, cant_ceros), mode='constant')
x3_zp = np.pad(x3, (0, cant_ceros), mode='constant')

# Eje de frecuencias con Zero Padding (0 a 500 Hz con paso de 0.1 Hz)
ff_zp = np.fft.rfftfreq(N_zp, 1/fs)

# Normalizamos por N (la cantidad de muestras de senal activa) para mantener 0 dB como 1 W
X1_zp = np.fft.rfft(x1_zp) / N
X2_zp = np.fft.rfft(x2_zp) / N
X3_zp = np.fft.rfft(x3_zp) / N

X1_zp_dB = 10 * np.log10(2 * np.abs(X1_zp)**2 + 1e-15)
X2_zp_dB = 10 * np.log10(2 * np.abs(X2_zp)**2 + 1e-15)
X3_zp_dB = 10 * np.log10(2 * np.abs(X3_zp)**2 + 1e-15)

# Grafico 2: Inciso c - Comparacion Zero Padding vs Original
fig, axs = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

# Subplot 1: k = N/4
axs[0].plot(ff_zp, X1_zp_dB, color='#FF007F', label='Con Zero Padding (10*N) - Continuo', lw=1.2)
axs[0].plot(ff_pos, X1_dB, 'ko', markersize=4, label='Sin Zero Padding (N) - Muestras')
axs[0].set_title(f'k = N/4 (f0 = {f1:.2f} Hz) - Coherente')
axs[0].set_ylabel('Potencia [dB]')
axs[0].set_ylim(-60, 5)
axs[0].grid(True, alpha=0.4)
axs[0].legend(loc='upper right', fontsize=9)

# Subplot 2: k = N/4 + 0.25
axs[1].plot(ff_zp, X2_zp_dB, color='#00A8FF', label='Con Zero Padding (10*N) - Continuo', lw=1.2)
axs[1].plot(ff_pos, X2_dB, 'ko', markersize=4, label='Sin Zero Padding (N) - Muestras')
axs[1].set_title(f'k = N/4 + 0.25 (f0 = {f2:.2f} Hz) - Desintonia intermedia')
axs[1].set_ylabel('Potencia [dB]')
axs[1].set_ylim(-60, 5)
axs[1].grid(True, alpha=0.4)
axs[1].legend(loc='upper right', fontsize=9)

# Subplot 3: k = N/4 + 0.5
axs[2].plot(ff_zp, X3_zp_dB, color='#00FF80', label='Con Zero Padding (10*N) - Continuo', lw=1.2)
axs[2].plot(ff_pos, X3_dB, 'ko', markersize=4, label='Sin Zero Padding (N) - Muestras')
axs[2].set_title(f'k = N/4 + 0.5 (f0 = {f3:.2f} Hz) - Maxima desintonia')
axs[2].set_xlabel('Frecuencia [Hz]')
axs[2].set_ylabel('Potencia [dB]')
axs[2].set_xlim(235, 265)   # Zoom alrededor de 250 Hz
axs[2].set_ylim(-60, 5)
axs[2].grid(True, alpha=0.4)
axs[2].legend(loc='upper right', fontsize=9)

fig.suptitle('Inciso c: Efecto de Zero Padding (Interpolacion de la DTFT)', fontsize=13, y=0.99)
plt.tight_layout()
plt.show()

**Análisis de la Figura 2 (Zero Padding):**

En estos tres gráficos se superpone la curva suave obtenida mediante zero padding con los puntos negros de la FFT original sin ceros agregados.

Lo primero que resalta es que la curva continua pasa exactamente por arriba de todos y cada uno de los puntos originales, lo que confirma que agregar ceros no altera los valores calculados originalmente sino que calcula puntos intermedios.

En el caso de $250.5\text{ Hz}$, se observa con claridad lo que sucedía en el inciso a: la cima real del lóbulo principal estaba ubicada justo en $250.5\text{ Hz}$ alcanzando los $0\text{ dB}$, pero la FFT original solo tomaba muestras en $250$ y $251\text{ Hz}$ a los costados del pico. Con la grilla más fina del zero padding ahora sí existe un bin exactamente en $250.5\text{ Hz}$, lo que permite recuperar visualmente el valor real de amplitud de la señal.

## 4. Discusión de Resultados

El análisis de estos tres experimentos permite comprender cómo interactúan la duración finita de una señal, el muestreo de la FFT y la visualización de su espectro.

La razón por la cual dos senoidales con frecuencias separadas por apenas fracciones de Hertz presentan aspectos visuales tan diferentes se debe a la naturaleza discreta de la DFT. La transformada de Fourier de una senoidal truncada en el tiempo por una ventana rectangular es en realidad una función continua con forma de sinc centrada en la frecuencia de la señal. Cuando calculamos la DFT, estamos tomando muestras de esa función continua cada $\Delta f = 1\text{ Hz}$. Si la frecuencia de la señal coincide con un múltiplo entero de $\Delta f$, todas las muestras de la DFT coinciden con los cruces por cero de los lóbulos laterales de la sinc, salvo la muestra central que captura el lóbulo principal; por eso el espectro parece un impulso ideal y sin desparramo. En cambio, si la señal está desfasada apenas $0.25\text{ Hz}$ o $0.5\text{ Hz}$, las muestras de la DFT ya no caen en los cruces por cero, haciendo visibles las laderas de todos los lóbulos laterales y provocando el desparramo en los bines vecinos.

La identidad de Parseval cumple un rol fundamental para interpretar este fenómeno, ya que demuestra que la energía no se destruye por efecto de la desintonía. Aunque el valor del bin central disminuya, esa potencia faltante se encuentra distribuida exactamente en los lóbulos secundarios, conservando en todo momento el valor total de $1\text{ W}$.

Por último, la aplicación de zero padding deja en evidencia que la aparente caída de amplitud que se observaba en el caso desintonizado era una limitación de muestreo frecuencial de la FFT y no una pérdida real de la señal. Al aumentar ficticiamente la cantidad de puntos de cálculo, la grilla se vuelve lo suficientemente densa como para muestrear la cima verdadera del lóbulo en $250.5\text{ Hz}$ y mostrar la forma completa de la transformada continua.

## 5. Conclusiones

1. El desparramo espectral es una consecuencia inevitable de observar una señal en un intervalo de tiempo finito cuando su frecuencia no coincide exactamente con un bin entero de la FFT. Esto genera una pérdida aparente de nivel en el pico y la aparición de lóbulos secundarios.
2. Se verificó experimentalmente la identidad de Parseval, comprobando que la potencia total calculada en el tiempo y en la frecuencia es exactamente de $1\text{ W}$ para las tres frecuencias evaluadas, lo que confirma que la energía total de la señal se conserva independientemente de la desintonía.
3. La técnica de zero padding demostró ser una herramienta muy eficaz para interpolar el espectro en frecuencia. Si bien no incrementa la resolución física del sistema para diferenciar tonos cercanos, permite visualizar la forma continua de la transformada y recuperar con precisión la amplitud real de señales que caen entre los bines de la FFT.